In [0]:
# Create a dummy dataset to test your engine
data = [("Energy_Project", 2026), ("Master_Students", 2)]
df = spark.createDataFrame(data, ["Project_Name", "Team_Size"])
display(df)

In [0]:
%sql
-- for creating a new catalog for the project and layers as well
-- 1. Create the Main Project Catalog
-- CREATE CATALOG IF NOT EXISTS energy_intelligence;

-- -- 2. Create the Medallion Schemas (Layers)
-- CREATE SCHEMA IF NOT EXISTS energy_intelligence.bronze 
--   COMMENT 'Raw data exactly as it arrived (landing zone)';

-- CREATE SCHEMA IF NOT EXISTS energy_intelligence.silver 
--   COMMENT 'Cleaned, filtered, and joined data';

-- CREATE SCHEMA IF NOT EXISTS energy_intelligence.gold 
--   COMMENT 'Aggregated data ready for PowerBI or ML models';

-- -- 3. Set this as your default so you don't have to type the full name every time
-- USE CATALOG energy_intelligence;

-- select * from parquet.`/Volumes/energy_intelligence/bronze/landing_zone/data/weather_data/`


In [0]:
from pyspark.sql.functions import col, to_date, from_unixtime

# 1. Define the schema manually (Use the names of your actual columns)
# If you don't know all the column names, run spark.read.parquet(path).columns first
custom_schema = "date LONG, temperature_2m FLOAT, cloud_cover FLOAT, wind_speed_100m FLOAT, surface_pressure FLOAT,wind_gusts_10m FLOAT, wind_direction_100m FLOAT, direct_normal_irradance FLOAT"

# 1. Process Hourly (Convert nanoseconds and add date-only bridge)
hourly_df = (spark.read
             .format("parquet")
             .schema(custom_schema)
             .load("/Volumes/energy_intelligence/bronze/landing_zone/data/weather_data/hourly/")
    .withColumn("timestamp", (col("date") / 1e9).cast("timestamp"))
    .withColumn("date_only", to_date((col("date") / 1e9).cast("timestamp"))) # The bridge column
)
display(hourly_df)
# 2. Process Daily (Ensure sunrise/sunset are proper timestamps)
custom_schema_2 = "date LONG, sunrise LONG, sunset LONG"
daily_df = (spark.read
            .format("parquet")
            .schema(custom_schema_2)
            .load("/Volumes/energy_intelligence/bronze/landing_zone/data/weather_data/daily/")  
    .withColumn("date_only", to_date(from_unixtime(col("date") / 1e9))) # Match the bridge column name
    .withColumn("sunrise", from_unixtime(col("sunrise") ).cast("timestamp"))
    .withColumn("sunset", from_unixtime(col("sunset") ).cast("timestamp"))
)
display(daily_df)

# 3. Save as separate Silver tables
hourly_df.write.mode("overwrite").saveAsTable("energy_intelligence.silver.weather_hourly")
daily_df.write.mode("overwrite").saveAsTable("energy_intelligence.silver.weather_daily")



In [0]:
%sql
-- creates a table in the gold schema
CREATE OR REPLACE TABLE energy_intelligence.gold.weather_enriched AS
SELECT 
  h.*, 
  d.sunrise, 
  d.sunset
FROM energy_intelligence.silver.weather_hourly h
LEFT JOIN energy_intelligence.silver.weather_daily d
  ON h.date_only = d.date_only;